# UniFest AI – College Events & Clubs Assistant

**Agentic AI – Day 2 (Alternative Example)**

A conversational agent for college events, clubs, registration guidance, and event-related complaints. Run in Google Colab and add your Groq API key in Colab Secrets as `groq_api_key`.

## 1. Install libraries

In [ ]:
!pip install -q openai ipywidgets

## 2. Connect to Groq API

In [ ]:
from openai import OpenAI
from google.colab import userdata

api_key = userdata.get("groq_api_key")
client = OpenAI(api_key=api_key, base_url="https://api.groq.com/openai/v1")
MODEL_NAME = "openai/gpt-oss-120b"
print("UniFest AI Connected!")

## 3. Define sample college information

In [ ]:
EVENT_INFO = {
    "tech_fest": "InnovateX – 15 December 2026, Main Auditorium",
    "cultural_fest": "Rhythm – 20 December 2026, Open-Air Theatre",
    "coding_contest": "CodeSprint – 10 December 2026, Computer Lab 2",
    "registration": "Register through the student portal",
    "registration_deadline": "Three days before each event",
    "coding_club": "CodeCraft Club – meets every Wednesday at 4 PM",
    "robotics_club": "RoboSphere Club – meets every Friday at 3 PM",
    "event_coordinator": "Student Activity Centre, Room A-12"
}
print(EVENT_INFO)

## 4. Create the system prompt

In [ ]:
import json

SYSTEM_PROMPT = f"""
ROLE:
You are UniFest AI, a friendly college events and clubs assistant.

AVAILABLE FACTS:
{json.dumps(EVENT_INFO)}

RULES:
1. Answer only using the available event and club information.
2. Never invent event dates, venues, deadlines, or registration links.
3. If information is missing, direct the student to the Student Activity Centre.
4. Keep answers short, clear, and student-friendly.
5. Use Telugu if the student requests it.
6. Never claim a student is registered unless a registration system confirms it.
"""

print("System prompt created!")

## 5. Classify the student's intent

In [ ]:
def classify_intent(message: str) -> str:
    messages = [
        {"role": "system", "content":
         "Classify into exactly one category: event_info, registration, club_info, complaint. Return only the category."},
        {"role": "user", "content": "When is the coding contest?"},
        {"role": "assistant", "content": "event_info"},
        {"role": "user", "content": "How do I register for InnovateX?"},
        {"role": "assistant", "content": "registration"},
        {"role": "user", "content": "Tell me about the robotics club."},
        {"role": "assistant", "content": "club_info"},
        {"role": "user", "content": "The registration page is not opening."},
        {"role": "assistant", "content": "complaint"},
        {"role": "user", "content": message}
    ]
    response = client.chat.completions.create(
        model=MODEL_NAME, temperature=0, messages=messages
    )
    return response.choices[0].message.content.strip().lower()

print(classify_intent("When will the cultural fest happen?"))

## 6. Extract complaint details as JSON

In [ ]:
def extract_event_complaint(message: str) -> dict:
    prompt = """
Extract event-related complaint details.
Return only valid JSON with these keys:
{
  "student_name": "string or null",
  "event_name": "string or null",
  "issue": "string",
  "location": "string or null",
  "priority": "low, medium or high"
}
Do not invent missing information. Use null when unknown.
"""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0,
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": message}
        ]
    )
    try:
        return json.loads(response.choices[0].message.content)
    except json.JSONDecodeError:
        return {"error": "The model did not return valid JSON"}

sample = (
    "Hi, I am Ananya from CSE. The microphone is not working "
    "at the InnovateX rehearsal in the Main Auditorium. Our practice starts in one hour."
)
result = extract_event_complaint(sample)
print(json.dumps(result, indent=2))

## 7. Create a logging decorator

In [ ]:
import functools
import time

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        print("Running:", func.__name__)
        result = func(*args, **kwargs)
        print("Execution time:", round(time.time() - start, 4), "seconds")
        return result
    return wrapper

@log_call
def check_registration_deadline(days_before_event: int) -> dict:
    if days_before_event < 0:
        return {"error": "The event date has already passed."}
    return {
        "days_remaining": days_before_event,
        "registration_open": days_before_event >= 3
    }

print(check_registration_deadline(5))

## 8. Build the agent with conversation memory

In [ ]:
class UniFestAgent:
    def __init__(self, system_prompt, max_turns=5):
        self.system_prompt = system_prompt
        self.max_turns = max_turns
        self.reset()

    def reset(self):
        self.history = [{"role": "system", "content": self.system_prompt}]

    def chat(self, message: str) -> str:
        if not message.strip():
            return "Please enter a message."

        self.history.append({"role": "user", "content": message})
        messages = [self.history[0]] + self.history[1:][-(self.max_turns * 2):]

        try:
            response = client.chat.completions.create(
                model=MODEL_NAME, messages=messages, temperature=0.3
            )
            reply = response.choices[0].message.content or ""
        except Exception as e:
            print("API error:", e)
            reply = "Sorry, UniFest AI is temporarily unavailable."

        self.history.append({"role": "assistant", "content": reply})
        return reply

agent = UniFestAgent(SYSTEM_PROMPT)
print("UniFest Agent Ready!")

## 9. Test the agent and its memory

In [ ]:
print(agent.chat("Hi, my name is Ananya."))
print(agent.chat("When is InnovateX?"))
print(agent.chat("Where will it take place?"))

## 10. Create an interactive chat interface

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

chat_output = widgets.Output()
message_input = widgets.Text(
    placeholder="Ask UniFest AI about events or clubs...",
    layout=widgets.Layout(width="75%")
)
send_button = widgets.Button(description="Send", button_style="primary")
reset_button = widgets.Button(description="Reset Chat", button_style="warning")

def send_message(button):
    message = message_input.value.strip()
    if not message:
        return
    message_input.value = ""
    with chat_output:
        print("You:", message)
        print("UniFest AI:", agent.chat(message))
        print("-" * 55)

def reset_chat(button):
    agent.reset()
    with chat_output:
        clear_output()
        print("Chat history reset.")

send_button.on_click(send_message)
reset_button.on_click(reset_chat)
message_input.on_submit(send_message)

display(widgets.HTML("<h2>🎉 UniFest AI – Events & Clubs Assistant</h2>"))
display(chat_output)
display(widgets.HBox([message_input, send_button, reset_button]))

## 11. Try these sample questions
1. When is the InnovateX tech fest?
2. Where will the coding contest take place?
3. How can I register for an event?
4. What is the registration deadline?
5. When does the RoboSphere Club meet?
6. The sound system is not working during rehearsal. Can I report it?

## Concepts demonstrated
- **Groq API:** Connects to the language model.
- **System prompt:** Defines the agent's role and boundaries.
- **Intent classification:** Identifies the student's request.
- **JSON extraction:** Converts complaints into structured data.
- **Decorator:** Logs function execution time.
- **Agent class:** Maintains conversation history.
- **ipywidgets:** Provides an interactive chat interface.

**Note:** This is a learning prototype using sample data. It does not connect to a real event registration database.